# Exploratory Data Analysis (EDA)

This notebook contains the exploratory analysis for the Financial Needs estimation project. It covers data quality checks, distribution diagnostics, correlation analysis, and dimensionality reduction — all of which informed our later modelling decisions.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# INSTALL DEPENDENCIES (run this cell once)
# ═══════════════════════════════════════════════════════════════
import subprocess, sys

packages = ['xgboost', 'lightgbm', 'shap', 'imbalanced-learn', 'xlrd']
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print("✅ All packages installed.")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# IMPORTS
# ═══════════════════════════════════════════════════════════════
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import (chi2_contingency, mannwhitneyu, spearmanr,
                          pearsonr, ks_2samp, shapiro, kruskal,
                          pointbiserialr, normaltest)

# Sklearn
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                      RandomizedSearchCV, cross_val_score,
                                      cross_val_predict, learning_curve)
from sklearn.preprocessing import (StandardScaler, MinMaxScaler,
                                    PowerTransformer, LabelEncoder)
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.calibration import calibration_curve, CalibratedClassifierCV

# Sklearn Models
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                               AdaBoostClassifier, ExtraTreesClassifier,
                               BaggingClassifier, HistGradientBoostingClassifier,
                               StackingClassifier, VotingClassifier)
from sklearn.neural_network import MLPClassifier

# Sklearn Metrics
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, roc_curve,
                              confusion_matrix, classification_report,
                              precision_recall_curve, average_precision_score,
                              matthews_corrcoef, balanced_accuracy_score,
                              log_loss,ConfusionMatrixDisplay, RocCurveDisplay)
from sklearn.inspection import permutation_importance

# XGBoost & LightGBM
import xgboost as xgb
import lightgbm as lgb

# SHAP
import shap

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

# Plot style
sns.set_theme(style='whitegrid', palette='viridis', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 150

print("✅ All libraries loaded successfully.")
print(f"   PyTorch  : {torch.__version__}")
print(f"   XGBoost  : {xgb.__version__}")
print(f"   LightGBM : {lgb.__version__}")
print(f"   SHAP     : {shap.__version__}")


## Data Loading

We load the three sheets from the Excel workbook — client-level data, product catalogue, and metadata — and apply a minor column-name fix.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# DATA LOADING
# ═══════════════════════════════════════════════════════════════
# ---- Adjust this path for your environment ----
# Google Colab with Drive:

# Local / uploaded file:
FILE_PATH = 'Dataset2_Needs.xls'  # <-- Put your .xls file in the same folder as this notebook

needs_df    = pd.read_excel(FILE_PATH, sheet_name='Needs')
products_df = pd.read_excel(FILE_PATH, sheet_name='Products')
metadata_df = pd.read_excel(FILE_PATH, sheet_name='Metadata')

# Fix trailing space in column name
needs_df.rename(columns={'Income ': 'Income'}, inplace=True)

print(f"Clients dataset : {needs_df.shape[0]:,} rows × {needs_df.shape[1]} columns")
print(f"Products catalog : {products_df.shape[0]} products × {products_df.shape[1]} attributes")
print(f"\nColumns: {needs_df.columns.tolist()}")
needs_df.head()


In [ ]:
# ═══════════════════════════════════════════════════════════════
# DATA DICTIONARY & PRODUCT CATALOG
# ═══════════════════════════════════════════════════════════════
meta_dict = dict(zip(
    metadata_df['Metadata'].dropna().astype(str),
    metadata_df['Unnamed: 1'].dropna().astype(str)
))
print("DATA DICTIONARY:")
for var, desc in meta_dict.items():
    print(f"  {str(var):30s} → {desc}")

product_names = {
    1: 'Balanced Mutual Fund', 2: 'Income Conservative Unit-Linked',
    3: 'Fixed Income Mutual Fund', 4: 'Balanced High Dividend MF',
    5: 'Balanced Mutual Fund II', 6: 'Defensive Flex Unit-Linked',
    7: 'Aggressive Flex Unit-Linked', 8: 'Balanced Flex Unit-Linked',
    9: 'Cautious Alloc Segregated', 10: 'Fixed Income Segregated',
    11: 'Total Return Aggr Segregated'
}
products_df['Name'] = products_df['IDProduct'].map(product_names)
products_df['TypeLabel'] = products_df['Type'].map({1: 'Accumulation', 0: 'Income'})
print("\nPRODUCT CATALOG:")
print(products_df.to_string(index=False))


---
## 1. Descriptive Statistics & Data Quality

Before any visual or statistical analysis, we check the basics: dataset dimensions, missing values, duplicates, and summary statistics including skewness and kurtosis. This flags potential issues (e.g. heavy tails in monetary variables) early on.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# COMPREHENSIVE DATA QUALITY REPORT
# ═══════════════════════════════════════════════════════════════
df = needs_df.drop('ID', axis=1).copy()

print("=" * 70)
print("DATA QUALITY REPORT")
print("=" * 70)
print(f"  Total observations      : {len(df):,}")
print(f"  Missing values (total)  : {df.isnull().sum().sum()}")
print(f"  Duplicate rows          : {df.duplicated().sum()}")
print(f"  Memory usage            : {df.memory_usage(deep=True).sum() / 1024:.1f} KB")

print("\n" + "=" * 70)
print("DESCRIPTIVE STATISTICS (EXTENDED)")
print("=" * 70)
desc = df.describe().T
desc['skewness'] = df.skew()
desc['kurtosis'] = df.kurtosis()
desc['IQR'] = desc['75%'] - desc['25%']
desc['CV'] = desc['std'] / desc['mean']  # Coefficient of variation
print(desc.round(4).to_string())

# Data types
print("\nData Types:")
print(df.dtypes.to_string())


## 2. Target Variable Analysis — Class Balance

We examine the prevalence of each target (`IncomeInvestment`, `AccumulationInvestment`) and test whether the two targets are independent using a chi-squared test. Knowing the class balance up front is important because it determines whether raw accuracy is a reliable metric or whether we need to focus on precision, recall, and threshold tuning.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# TARGET CLASS BALANCE + CHI-SQUARED INDEPENDENCE TEST
# ═══════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, target in zip(axes[:2], ['IncomeInvestment', 'AccumulationInvestment']):
    counts = df[target].value_counts()
    ratio = counts[1] / len(df)
    colors = ['#3498db', '#e74c3c']
    bars = ax.bar(['No Need (0)', 'Has Need (1)'], counts.values,
                   color=colors, edgecolor='black', alpha=0.85)
    ax.set_title(f'{target}\nPrevalence: {ratio:.1%} | Imbalance ratio: {counts[0]/counts[1]:.2f}:1',
                 fontsize=11)
    ax.set_ylabel('Count')
    for bar, v in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, v + 30, str(v),
                ha='center', fontweight='bold')

# Cross-tabulation heatmap
ct = pd.crosstab(df['IncomeInvestment'], df['AccumulationInvestment'])
chi2, p, dof, expected = chi2_contingency(ct)
sns.heatmap(ct, annot=True, fmt='d', cmap='YlOrRd', ax=axes[2])
axes[2].set_title(f'Target Cross-Tab\nχ²={chi2:.2f}, p={p:.4f}')
axes[2].set_xlabel('AccumulationInvestment')
axes[2].set_ylabel('IncomeInvestment')

plt.tight_layout()
plt.show()

print(f"Chi-squared independence test: χ²={chi2:.2f}, p={p:.4e}")
print(f"→ Targets are {'associated' if p < 0.05 else 'independent'} (α=0.05)")
print(f"→ Separate binary classifiers (One-vs-All) are appropriate.")


**Figure explanation:**

**Bar chart — IncomeInvestment (left):** Shows how many clients have an income-type investment need (class 1) versus those who do not (class 0). The prevalence percentage and the imbalance ratio printed in the title tell us how skewed the split is. If, for example, only ~30 % of clients have the need, a naive classifier that always predicts "no need" would already reach 70 % accuracy — so accuracy alone is not a trustworthy metric for this target.

**Bar chart — AccumulationInvestment (centre):** Same logic applied to the accumulation-type need. Comparing the two charts side by side lets us see whether one target is harder (more imbalanced) than the other, which may require different threshold-tuning strategies later.

**Heatmap — Target cross-tabulation (right):** Each cell counts how many clients fall into a specific combination of the two targets (e.g. need both, need neither, need only one). The chi-squared statistic and p-value printed in the title test whether the two targets are statistically independent. A low p-value means they co-occur more (or less) often than chance would predict, but as long as the association is not extremely strong, modelling them as two separate binary classifiers remains a reasonable approach.

## 3. Feature Distributions — Histograms, Box Plots & Q-Q Plots

For each continuous feature we plot three complementary views: a histogram with KDE overlay (to see shape and skewness), a box plot (to spot outliers and spread), and a Q-Q plot (to check deviation from normality). We also run Shapiro-Wilk and D'Agostino tests to confirm visual impressions quantitatively.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# DISTRIBUTION ANALYSIS: Histograms + Box Plots + Q-Q Plots
# ═══════════════════════════════════════════════════════════════
continuous_features = ['Age', 'FamilyMembers', 'FinancialEducation',
                       'RiskPropensity', 'Income', 'Wealth']

fig, axes = plt.subplots(len(continuous_features), 3, figsize=(18, 4*len(continuous_features)))

for i, feat in enumerate(continuous_features):
    # Histogram + KDE
    ax = axes[i, 0]
    sns.histplot(df[feat], kde=True, ax=ax, color='steelblue', bins=40, alpha=0.7)
    ax.axvline(df[feat].mean(), color='red', ls='--', lw=1.2, label=f'μ={df[feat].mean():.2f}')
    ax.axvline(df[feat].median(), color='green', ls=':', lw=1.2, label=f'med={df[feat].median():.2f}')
    ax.legend(fontsize=8)
    ax.set_title(f'{feat} — Distribution (skew={df[feat].skew():.2f})')

    # Box Plot
    ax2 = axes[i, 1]
    sns.boxplot(x=df[feat], ax=ax2, color='lightcoral')
    Q1, Q3 = df[feat].quantile(0.25), df[feat].quantile(0.75)
    IQR = Q3 - Q1
    outliers = ((df[feat] < Q1 - 1.5*IQR) | (df[feat] > Q3 + 1.5*IQR)).sum()
    ax2.set_title(f'{feat} — Box Plot (outliers: {outliers})')

    # Q-Q Plot
    ax3 = axes[i, 2]
    stats.probplot(df[feat], dist='norm', plot=ax3)
    ax3.set_title(f'{feat} — Q-Q Plot')

plt.tight_layout()
plt.show()

# Normality tests
print("\nNormality Tests (Shapiro-Wilk, subsample n=500):")
print("-" * 60)
for feat in continuous_features:
    sample = df[feat].sample(500, random_state=42)
    w_stat, w_p = shapiro(sample)
    k2, k_p = normaltest(sample)
    print(f"  {feat:25s}: Shapiro W={w_stat:.4f} (p={w_p:.2e}) | "
          f"D'Agostino K²={k2:.2f} (p={k_p:.2e}) → "
          f"{'Normal' if w_p > 0.05 and k_p > 0.05 else 'Non-normal'}")


**Figure explanation — one row per feature:**

**Histogram + KDE (left column):** The blue histogram shows how values are spread across ranges, and the KDE curve smooths it into a continuous shape. The red dashed line marks the mean, the green dotted line marks the median. When these two lines sit close together the distribution is roughly symmetric; when the mean is pulled to the right of the median, the variable is right-skewed. The skewness value printed in the title quantifies this. For `Age`, `FamilyMembers`, `FinancialEducation`, and `RiskPropensity` we typically see moderate or near-symmetric shapes. For `Income` and `Wealth`, the histogram shows a long right tail — most clients cluster at lower values while a smaller group has much higher amounts.

**Box plot (centre column):** The box spans the interquartile range (IQR, 25th to 75th percentile) and the whiskers extend to 1.5 × IQR. Points beyond the whiskers are flagged as outliers, and the count is shown in the title. A large number of outliers — common for `Income` and `Wealth` — confirms that extreme values are present and may need to be handled through transformations or by choosing models that are robust to them.

**Q-Q plot (right column):** Each point compares an observed data quantile against the theoretical quantile of a normal distribution. If the points follow the red diagonal closely, the variable is approximately normal. Upward curvature at the right end indicates a heavier right tail than normal, which is the typical pattern for monetary variables.

**Normality tests (printed output):** Shapiro-Wilk and D'Agostino K² tests formalise what the Q-Q plots show visually. A p-value below 0.05 means we reject the normality assumption. This is relevant because certain statistical tests and model families assume roughly normal inputs — when that assumption fails, non-parametric methods or prior transformations are more appropriate.

## 4. Monetary Variable Transformations

`Income` and `Wealth` exhibit strong right skew and fat tails. We compare three representations — raw, log1p, and Yeo-Johnson — to find a transformation that stabilises variance and reduces skewness without losing interpretability.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# WEALTH & INCOME TRANSFORMATION COMPARISON
# ═══════════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 3, figsize=(18, 8))

for row, feat in enumerate(['Wealth', 'Income']):
    data = df[feat]

    # Raw
    sns.histplot(data, kde=True, ax=axes[row, 0], color='steelblue', bins=50)
    axes[row, 0].set_title(f'{feat} — Raw\nskew={data.skew():.2f}, kurt={data.kurtosis():.2f}')

    # Log1p
    log_data = np.log1p(data)
    sns.histplot(log_data, kde=True, ax=axes[row, 1], color='darkorange', bins=50)
    axes[row, 1].set_title(f'{feat} — log1p\nskew={log_data.skew():.2f}')

    # Yeo-Johnson
    pt = PowerTransformer(method='yeo-johnson')
    yj_data = pt.fit_transform(data.values.reshape(-1, 1)).flatten()
    sns.histplot(yj_data, kde=True, ax=axes[row, 2], color='forestgreen', bins=50)
    axes[row, 2].set_title(f'{feat} — Yeo-Johnson\nskew={pd.Series(yj_data).skew():.2f}')

plt.suptitle('Transformation Comparison for Fat-Tailed Monetary Variables', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()
print("→ log1p provides good normalization; Yeo-Johnson achieves near-zero skewness.")
print("→ We will use log1p in feature engineering (interpretable) and StandardScaler for model input.")


**Figure explanation — one row per monetary variable (Wealth, Income):**

**Raw distribution (left column):** Shows the original, untransformed values. Both variables exhibit strong positive skew and high kurtosis — the bulk of observations sit in a narrow low range, with a long tail stretching toward much higher values. This shape can cause distance-based models (e.g. KNN, SVM) and gradient-based optimisers to be dominated by extreme values.

**log1p transformation (centre column):** Applying `log(1 + x)` compresses the right tail substantially. The resulting distribution is much closer to bell-shaped, and the skewness value drops considerably. Because the log function is monotonic and interpretable (a one-unit change in log-income corresponds to a multiplicative change in raw income), this is a practical choice for feature engineering.

**Yeo-Johnson transformation (right column):** This is a parametric power transformation that searches for the lambda value minimising skewness. It typically achieves near-zero skewness, producing an almost perfectly symmetric distribution. However, the transformed values lose direct interpretability — they live on an abstract scale rather than a meaningful monetary one.

**Conclusion:** `log1p` offers the best trade-off between normalisation and interpretability, so we adopt it for engineered features. For model input, we additionally apply `StandardScaler` to centre and scale the transformed values.

## 5. Correlation Analysis

We compute both Pearson (linear) and Spearman (monotonic/rank) correlation matrices. Comparing the two helps identify relationships that are real but non-linear — a common situation with financial variables.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CORRELATION HEATMAPS (PEARSON + SPEARMAN)
# ═══════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

mask = np.triu(np.ones_like(df.corr(), dtype=bool))

sns.heatmap(df.corr(method='pearson'), annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=axes[0], vmin=-1, vmax=1, mask=mask, square=True)
axes[0].set_title('Pearson Correlation', fontsize=12)

sns.heatmap(df.corr(method='spearman'), annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=axes[1], vmin=-1, vmax=1, mask=mask, square=True)
axes[1].set_title('Spearman Rank Correlation', fontsize=12)

plt.tight_layout()
plt.show()


**Figure explanation:**

**Pearson correlation (left):** Measures the strength of *linear* relationships between each pair of variables. Values near +1 or −1 indicate a strong linear association; values near 0 suggest no linear relationship. Look for which features correlate most with the two targets (`IncomeInvestment`, `AccumulationInvestment`). Also pay attention to inter-feature correlations — if two features are highly correlated with each other (e.g. `Income` and `Wealth`), they carry overlapping information, which can affect model stability.

**Spearman rank correlation (right):** Measures *monotonic* (rank-based) relationships instead of strictly linear ones. This is more robust to outliers and captures associations where one variable consistently increases with another even if the relationship is curved. When a pair shows low Pearson but noticeably higher Spearman correlation, that signals a real but non-linear relationship — meaning tree-based or other flexible models may exploit it better than a linear classifier.

**Reading both together:** Comparing the two matrices side by side helps distinguish truly weak relationships (low in both) from relationships that are real but non-linear (low Pearson, higher Spearman). The upper triangle is masked to avoid visual redundancy since correlation matrices are symmetric.

## 6. Point-Biserial Correlations & Mann-Whitney U Tests

To go beyond visual patterns, we measure the strength and significance of each feature's association with the binary targets. Point-biserial correlation gives effect size; Mann-Whitney U tests whether the distributions of a feature differ significantly between positive and negative cases.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# POINT-BISERIAL CORRELATIONS + MANN-WHITNEY U TESTS
# ═══════════════════════════════════════════════════════════════
features_list = ['Age', 'Gender', 'FamilyMembers', 'FinancialEducation',
                 'RiskPropensity', 'Income', 'Wealth']

print("=" * 80)
print("POINT-BISERIAL CORRELATIONS WITH TARGETS")
print("=" * 80)
for target in ['IncomeInvestment', 'AccumulationInvestment']:
    print(f"\n  {target}:")
    for feat in features_list:
        r, p = pointbiserialr(df[target], df[feat])
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        print(f"    {feat:25s}: r={r:+.4f}, p={p:.4e} {sig}")

print("\n" + "=" * 80)
print("MANN-WHITNEY U TESTS (Y=0 vs Y=1)")
print("=" * 80)
for target in ['IncomeInvestment', 'AccumulationInvestment']:
    print(f"\n  {target}:")
    for feat in features_list:
        g0 = df.loc[df[target] == 0, feat]
        g1 = df.loc[df[target] == 1, feat]
        u_stat, p = mannwhitneyu(g0, g1, alternative='two-sided')
        eff_r = 1 - (2*u_stat) / (len(g0)*len(g1))  # rank-biserial
        print(f"    {feat:25s}: U={u_stat:>12,.0f}, p={p:.4e}, r_rb={eff_r:+.3f} "
              f"{'***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'ns'}")


**Output explanation:**

**Point-biserial correlations:** For each feature–target pair, the point-biserial coefficient `r` measures the strength and direction of linear association between a continuous feature and a binary target. Values close to 0 mean the feature's average is nearly the same for both target classes; larger absolute values mean the feature shifts meaningfully between classes 0 and 1. The significance markers (`***` for p < 0.001, `**` for p < 0.01, `*` for p < 0.05, `ns` for not significant) indicate whether the observed association is statistically reliable given the sample size. Features with strong, significant correlations to a target are natural candidates for the model to leverage.

**Mann-Whitney U tests:** This is a non-parametric alternative that compares the full distributions (not just means) of a feature between the two target groups. The U statistic and p-value test whether one group tends to have systematically higher values than the other. The rank-biserial correlation `r_rb` is an effect-size measure: values near 0 mean the two groups are nearly identical in rank, while larger absolute values indicate meaningful separation. Features where both the point-biserial and Mann-Whitney tests agree on significance give us the most confidence that a real relationship exists.

## 7. Feature Interactions — Violin Plots by Target

We compare the distribution of key features across target classes using violin plots overlaid with strip plots. This reveals whether a feature creates separation between classes or whether the two groups overlap heavily.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# VIOLIN PLOTS — FEATURE DISTRIBUTIONS BY TARGET
# ═══════════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 4, figsize=(22, 10))

for row, target in enumerate(['IncomeInvestment', 'AccumulationInvestment']):
    for col, feat in enumerate(['Age', 'RiskPropensity', 'Income', 'Wealth']):
        ax = axes[row, col]
        sns.violinplot(x=df[target], y=df[feat], ax=ax,
                       palette=['#3498db', '#e74c3c'], inner='quartile',
                       cut=0, bw_adjust=0.8)
        # Overlay strip plot for granularity
        sns.stripplot(x=df[target], y=df[feat], ax=ax,
                      color='black', alpha=0.05, size=1)
        ax.set_title(f'{feat} by {target}', fontsize=10)
        ax.set_xlabel('')

plt.suptitle('Feature Distributions Segmented by Investment Need', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()


**Figure explanation — 2×4 grid (rows = targets, columns = features):**

**Top row — IncomeInvestment:** Each subplot shows how a feature (`Age`, `RiskPropensity`, `Income`, `Wealth`) is distributed for clients without an income investment need (class 0, blue) versus clients with one (class 1, red). The violin width represents density — wider sections mean more clients sit at that value. The inner dashed lines mark the quartiles (25th, 50th, 75th percentile). If the two violins have clearly different medians or shapes, the feature carries discriminative power for this target. Overlapping violins mean the feature alone does not strongly separate the classes.

**Bottom row — AccumulationInvestment:** Same layout, but now segmented by the accumulation target. Comparing the two rows reveals whether the same feature is useful for both targets or only one — for example, `Wealth` might shift noticeably for `AccumulationInvestment` but not for `IncomeInvestment`, or vice versa.

**Strip plot overlay (black dots):** The faint individual data points layered on top of each violin give a sense of raw data density, especially in sparse regions where the KDE smoothing in the violin might overstate density. Clusters of dots at extreme values confirm the presence of outliers already detected in the box plots.

## 8. Gender × Target Interaction

We check whether need prevalence varies by gender, and whether family size differs across combined target groups (Kruskal-Wallis test). These interactions can reveal whether demographic segmentation adds predictive value.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# GENDER × TARGET INTERACTION ANALYSIS
# ═══════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, target in zip(axes, ['IncomeInvestment', 'AccumulationInvestment']):
    ct = pd.crosstab(df['Gender'], df[target], normalize='index') * 100
    ct.plot(kind='bar', ax=ax, color=['#3498db', '#e74c3c'],
            edgecolor='black', alpha=0.85)
    ax.set_title(f'Need Prevalence by Gender — {target}')
    ax.set_xlabel('Gender (0=Male, 1=Female)')
    ax.set_ylabel('Percentage (%)')
    ax.set_xticklabels(['Male', 'Female'], rotation=0)
    ax.legend(['No Need', 'Has Need'])

plt.tight_layout()
plt.show()

# Kruskal-Wallis test: do family members differ by combined target group?
df['NeedGroup'] = df['IncomeInvestment'].astype(str) + '_' + df['AccumulationInvestment'].astype(str)
groups = [g['FamilyMembers'].values for _, g in df.groupby('NeedGroup')]
h_stat, h_p = kruskal(*groups)
print(f"Kruskal-Wallis test (FamilyMembers across need groups): H={h_stat:.2f}, p={h_p:.4e}")
df.drop('NeedGroup', axis=1, inplace=True)


**Figure explanation:**

**Bar chart — IncomeInvestment by Gender (left):** Shows the percentage of male (Gender = 0) and female (Gender = 1) clients who do and do not have an income investment need. If both genders have nearly the same proportion of "Has Need" (red bar), gender is not a strong standalone predictor for this target. If one gender shows a noticeably higher red bar, that suggests a demographic-level difference worth exploring as a feature or interaction term.

**Bar chart — AccumulationInvestment by Gender (right):** Same comparison for the accumulation target. Again, look for whether the red/blue split changes meaningfully between male and female clients. Together, the two charts tell us whether gender plays a different role for each of the two financial needs.

**Kruskal-Wallis test (printed output):** This test groups clients into four combined need categories (no need for either, income only, accumulation only, both) and checks whether `FamilyMembers` differs across these groups. A significant result (low p-value) would suggest that household size is associated with the type of financial need a client has, which could be useful as an interaction feature in the model.

## 9. Dimensionality Reduction — PCA

As a final diagnostic, we apply PCA to the standardised feature matrix. The scree plot shows how variance is distributed across components, and the 2D projections let us visually assess whether the target classes form separable clusters in reduced space. PCA here is an exploratory tool — not a modelling decision.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# PCA ANALYSIS + SCREE PLOT
# ═══════════════════════════════════════════════════════════════
X_viz = df[features_list].copy()
scaler_viz = StandardScaler()
X_scaled_viz = scaler_viz.fit_transform(X_viz)

# Full PCA
pca_full = PCA(random_state=42).fit(X_scaled_viz)

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Scree plot
ax = axes[0]
cumvar = np.cumsum(pca_full.explained_variance_ratio_)
ax.bar(range(1, len(pca_full.explained_variance_ratio_)+1),
       pca_full.explained_variance_ratio_, alpha=0.7, color='steelblue', label='Individual')
ax.plot(range(1, len(cumvar)+1), cumvar, 'ro-', label='Cumulative')
ax.axhline(y=0.95, color='gray', ls='--', alpha=0.5, label='95% threshold')
ax.set_xlabel('Principal Component')
ax.set_ylabel('Explained Variance Ratio')
ax.set_title('PCA Scree Plot')
ax.legend(fontsize=9)

# 2D PCA colored by each target
pca_2d = PCA(n_components=2, random_state=42)
X_pca = pca_2d.fit_transform(X_scaled_viz)

for ax, target in zip(axes[1:], ['IncomeInvestment', 'AccumulationInvestment']):
    scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=df[target],
                         cmap='RdYlBu', alpha=0.35, s=8, edgecolors='none')
    ax.set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.1%})')
    ax.set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.1%})')
    ax.set_title(f'PCA — {target}')
    plt.colorbar(scatter, ax=ax)

plt.tight_layout()
plt.show()

print(f"PCA explained variance (2 components): {pca_2d.explained_variance_ratio_.sum():.1%}")
n_95 = np.argmax(cumvar >= 0.95) + 1
print(f"Components for 95% variance: {n_95}")


**Figure explanation:**

**Scree plot (left):** Each blue bar shows the fraction of total variance captured by one principal component, and the red line tracks the cumulative sum. The grey dashed line marks the 95 % threshold. If the first two or three components already capture most of the variance, the effective dimensionality of the dataset is low — meaning the features contain a lot of redundant or correlated information. If variance is spread evenly across many components, each feature contributes unique information. The number of components needed to reach 95 % is printed below the figure.

**2D scatter — IncomeInvestment (centre):** Each point is one client, projected onto the first two principal components. Colour represents the target class. If the two colours form distinct regions or clusters, the classes are separable in the top-2 PCA directions — meaning even a simple linear classifier might work. If the colours are thoroughly mixed, linear separation in reduced dimensions is weak, and we should expect to need non-linear models (e.g. tree-based ensembles, neural networks) that can operate in the full feature space.

**2D scatter — AccumulationInvestment (right):** Same projection, coloured by the second target. Comparing the two scatter plots lets us see whether one target has better structure in PCA space than the other, which can foreshadow relative modelling difficulty.

**Overall takeaway:** PCA here is purely diagnostic. The scree plot tells us about feature redundancy, and the scatter plots give a quick visual sanity check of class separability. Neither replaces proper modelling — they simply inform expectations.

---
## Summary

Key takeaways from this EDA:

- Both targets are moderately imbalanced, so evaluation should go beyond accuracy (F1, AUC, precision-recall).
- `Income` and `Wealth` are heavily right-skewed; `log1p` is a good normalising transformation.
- Several features show statistically significant associations with the targets, though effect sizes are modest.
- PCA confirms that the feature space is relatively low-dimensional but classes are not linearly separable in 2D — non-linear models are warranted.

These findings directly inform the feature engineering, model selection, and evaluation strategy in the main modelling notebook.